In [ ]:

# ============================================================
# DATA QUALITY & CLEANING PROJECT
# Dataset: LendingClub Accepted 50K
# Local Windows / Jupyter Notebook - ONE CELL VERSION
# ============================================================

# 1. IMPORT LIBRARIES
import os
import warnings
import shutil
from pathlib import Path
from itertools import combinations

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 200)
pd.set_option("display.max_rows", 100)

# ============================================================
# 2. FILE PATHS
# ============================================================

PROJECT_DIR = r"G:\DEPI\Projects\Project\DEPI-ITC-ONL5-DA-Team3-G3-Projects\FinalProject"
INPUT_DIR = r"G:\DEPI\Projects\Project\DEPI-ITC-ONL5-DA-Team3-G3-Projects\FinalProject\data\Sampled"
CLEANED_DIR = r"G:\DEPI\Projects\Project\DEPI-ITC-ONL5-DA-Team3-G3-Projects\FinalProject\data\Cleaned"
REPORT_ROOT = r"G:\DEPI\Projects\Project\DEPI-ITC-ONL5-DA-Team3-G3-Projects\FinalProject\docs\Data Cleaning"
DATASET_REPORT_DIR = os.path.join(REPORT_ROOT, "Accepted")
BEFORE_DIR = os.path.join(DATASET_REPORT_DIR, "Before_Cleaning")
AFTER_DIR = os.path.join(DATASET_REPORT_DIR, "After_Cleaning")
REPORT_DIR = DATASET_REPORT_DIR

FILE_PATH = os.path.join(INPUT_DIR, "sample_accepted_2007_to_2018Q4.csv")
CLEANED_FILE = os.path.join(CLEANED_DIR, "accepted_cleaned.csv")
GUIDE_FILE = os.path.join(REPORT_ROOT, "Accepted_Cleaning_Guide.md")

for folder in [CLEANED_DIR, REPORT_ROOT, DATASET_REPORT_DIR, BEFORE_DIR, AFTER_DIR]:
    os.makedirs(folder, exist_ok=True)

# ============================================================
# 3. LOAD RAW DATA
# ============================================================

def load_data(file_path):
    if not os.path.exists(file_path):
        raise FileNotFoundError(
            f"File not found: {file_path}. "
            "Check that the sampled CSV exists in the configured data\\Sampled directory."
        )

    try:
        return pd.read_csv(file_path, low_memory=False)
    except UnicodeDecodeError:
        return pd.read_csv(
            file_path,
            encoding="cp1252",
            low_memory=False
        )

raw_df = load_data(FILE_PATH)
df = raw_df.copy(deep=True)

print("DATA LOADED SUCCESSFULLY")
print("Rows:", df.shape[0])
print("Columns:", df.shape[1])
display(df.head())

# ============================================================
# 4. BASIC DATA INFORMATION
# ============================================================

print("\nDATASET INFORMATION")
print("Shape:", df.shape)

print("\nData Types:")
display(df.dtypes.astype(str))

print("\nStatistical Description:")
display(df.describe(include="all").T)

print("\nDuplicate Rows:", int(df.duplicated().sum()))

print(
    "Memory Usage:",
    round(df.memory_usage(deep=True).sum() / 1024**2, 2),
    "MB"
)

# ============================================================
# 5. BASIC QUALITY PROFILE
# ============================================================

quality_profile = pd.DataFrame({
    "Column": df.columns,
    "Data_Type": df.dtypes.astype(str).values,
    "Missing_Count": df.isnull().sum().values,
    "Missing_Percentage": (
        df.isnull().mean().values * 100
    ).round(2),
    "Unique_Values": df.nunique(dropna=True).values
}).sort_values("Missing_Percentage", ascending=False)

quality_profile.to_csv(
    os.path.join(REPORT_DIR, "Quality_Profile.csv"),
    index=False
)

display(quality_profile)

# ============================================================
# OBJECTIVE 1: GRAPHS BEFORE CLEANING
# ============================================================

# ============================================================
# 6. NULL VALUES ANALYSIS
# ============================================================

null_report = pd.DataFrame({
    "Column": df.columns,
    "Null_Count": df.isnull().sum().values,
    "Null_Percentage": (
        df.isnull().mean().values * 100
    ).round(2)
}).sort_values("Null_Percentage", ascending=False)

null_report.to_csv(
    os.path.join(REPORT_DIR, "Null_Report_Before.csv"),
    index=False
)

display(null_report)

top_nulls = (
    null_report[null_report["Null_Count"] > 0]
    .head(40)
    .sort_values("Null_Percentage")
)

plt.figure(figsize=(13, 10))

if not top_nulls.empty:
    plt.barh(
        top_nulls["Column"],
        top_nulls["Null_Percentage"]
    )
    plt.title("Missing Values by Column - Before Cleaning")
    plt.xlabel("Missing Values (%)")
    plt.ylabel("Columns")
else:
    plt.text(0.5, 0.5, "No Missing Values Found",
             ha="center", va="center")
    plt.axis("off")

plt.tight_layout()
plt.savefig(
    os.path.join(BEFORE_DIR, "Null_Values.png"),
    dpi=160
)
plt.show()
plt.close()

# Missing values per row
row_missing_percentage = df.isnull().mean(axis=1) * 100

plt.figure(figsize=(10, 5))
plt.hist(row_missing_percentage, bins=30)
plt.title("Missing Values Per Row")
plt.xlabel("Missing Percentage")
plt.ylabel("Number of Rows")
plt.tight_layout()
plt.savefig(
    os.path.join(BEFORE_DIR, "Row_Missingness.png"),
    dpi=160
)
plt.show()
plt.close()

# ============================================================
# 7. CO-MISSING COLUMNS
# ============================================================

def analyze_comissing(data):
    columns = data.columns[data.isnull().any()].tolist()
    results = []

    for col1, col2 in combinations(columns, 2):
        missing1 = data[col1].isnull()
        missing2 = data[col2].isnull()

        both_missing = int((missing1 & missing2).sum())
        union = int((missing1 | missing2).sum())

        if both_missing >= 5 and union > 0:
            correlation = missing1.astype(int).corr(
                missing2.astype(int)
            )

            results.append({
                "Column_1": col1,
                "Column_2": col2,
                "Both_Missing": both_missing,
                "Jaccard_Similarity": both_missing / union,
                "Missing_Correlation": correlation
            })

    return pd.DataFrame(
        results,
        columns=[
            "Column_1",
            "Column_2",
            "Both_Missing",
            "Jaccard_Similarity",
            "Missing_Correlation"
        ]
    )

comissing_report = analyze_comissing(df)

comissing_report.to_csv(
    os.path.join(REPORT_DIR, "Co_Missing_Columns.csv"),
    index=False
)

display(comissing_report.head(50))

# ============================================================
# 8. OUTLIERS ANALYSIS
# ============================================================

OUTLIER_COLUMNS = [
    "Column",
    "Lower_Bound",
    "Upper_Bound",
    "Outlier_Count",
    "Outlier_Percentage",
    "Minimum",
    "Maximum"
]

def detect_outliers(data):
    numeric_data = data.select_dtypes(include=[np.number])
    results = []

    for column in numeric_data.columns:
        values = numeric_data[column].dropna()

        if len(values) < 10 or values.nunique() <= 2:
            continue

        q1 = values.quantile(0.25)
        q3 = values.quantile(0.75)
        iqr = q3 - q1

        if pd.isna(iqr):
            continue

        if iqr == 0:
            lower = values.quantile(0.01)
            upper = values.quantile(0.99)
        else:
            lower = q1 - 1.5 * iqr
            upper = q3 + 1.5 * iqr

        outlier_mask = (values < lower) | (values > upper)

        results.append({
            "Column": column,
            "Lower_Bound": lower,
            "Upper_Bound": upper,
            "Outlier_Count": int(outlier_mask.sum()),
            "Outlier_Percentage": float(outlier_mask.mean() * 100),
            "Minimum": values.min(),
            "Maximum": values.max()
        })

    return (
        pd.DataFrame(results, columns=OUTLIER_COLUMNS)
        .sort_values("Outlier_Percentage", ascending=False)
        .reset_index(drop=True)
    )

outlier_report = detect_outliers(df)

outlier_report.to_csv(
    os.path.join(REPORT_DIR, "Outliers_Report.csv"),
    index=False
)

display(outlier_report.head(50))

top_outliers = (
    outlier_report.head(25)
    .sort_values("Outlier_Percentage")
)

plt.figure(figsize=(12, 8))

if not top_outliers.empty:
    plt.barh(
        top_outliers["Column"],
        top_outliers["Outlier_Percentage"]
    )
    plt.title("Potential Outliers by Column")
    plt.xlabel("Outliers Percentage")
else:
    plt.text(0.5, 0.5, "No Eligible Numeric Columns",
             ha="center", va="center")
    plt.axis("off")

plt.tight_layout()
plt.savefig(
    os.path.join(BEFORE_DIR, "Outliers_Graph.png"),
    dpi=160
)
plt.show()
plt.close()

# ============================================================
# 9. INCONSISTENT CATEGORIES
# ============================================================

CATEGORY_COLUMNS = [
    "Column",
    "Normalized_Value",
    "Original_Variants",
    "Number_Of_Variants"
]

def find_category_variants(data):
    results = []

    categorical_columns = data.select_dtypes(
        include=["object", "string", "category"]
    ).columns

    for column in categorical_columns:
        values = data[column].dropna().astype(str)

        unique_count = values.nunique()

        if unique_count < 2 or unique_count > 100:
            continue

        normalized = values.str.strip().str.casefold()

        groups = pd.DataFrame({
            "Original": values.to_numpy(),
            "Normalized": normalized.to_numpy()
        })

        grouped = groups.groupby("Normalized")["Original"].agg(
            lambda x: sorted(set(x))
        )

        for key, variants in grouped.items():
            if len(variants) > 1:
                results.append({
                    "Column": column,
                    "Normalized_Value": key,
                    "Original_Variants": " || ".join(variants),
                    "Number_Of_Variants": len(variants)
                })

    return pd.DataFrame(results, columns=CATEGORY_COLUMNS)

category_report = find_category_variants(df)

category_report.to_csv(
    os.path.join(REPORT_DIR, "Category_Inconsistencies.csv"),
    index=False
)

display(category_report.head(100))

# ============================================================
# 10. ISSUE OVERLAP
# ============================================================

def analyze_issue_overlap(data, outliers, categories):
    flags = pd.DataFrame(index=data.index)

    # Null issue
    flags["Has_Null"] = data.isnull().any(axis=1)

    # Outlier issue
    outlier_rows = pd.Series(
        False,
        index=data.index,
        dtype=bool
    )

    if outliers is not None and "Column" in outliers.columns:
        for _, row in outliers.iterrows():
            column = row["Column"]

            if column not in data.columns:
                continue

            values = pd.to_numeric(
                data[column],
                errors="coerce"
            )

            lower = row["Lower_Bound"]
            upper = row["Upper_Bound"]

            if pd.notna(lower) and pd.notna(upper):
                outlier_rows |= (
                    values.notna() &
                    ((values < lower) | (values > upper))
                )

    flags["Has_Potential_Outlier"] = outlier_rows

    # Whitespace issue
    whitespace_rows = pd.Series(
        False,
        index=data.index,
        dtype=bool
    )

    if categories is not None and "Column" in categories.columns:
        for column in categories["Column"].dropna().unique():
            if column not in data.columns:
                continue

            values = data[column].astype("string")

            whitespace_rows |= (
                values.notna() &
                values.ne(values.str.strip())
            )

    flags["Has_Whitespace"] = whitespace_rows

    flags["Total_Issue_Types"] = flags[
        [
            "Has_Null",
            "Has_Potential_Outlier",
            "Has_Whitespace"
        ]
    ].sum(axis=1)

    return flags

issue_flags = analyze_issue_overlap(
    df,
    outlier_report,
    category_report
)

issue_flags.to_csv(
    os.path.join(REPORT_DIR, "Issue_Overlap.csv"),
    index=False
)

print("\nIssue Counts:")
display(
    issue_flags[
        ["Has_Null", "Has_Potential_Outlier", "Has_Whitespace"]
    ].sum()
)

plt.figure(figsize=(8, 5))

issue_counts = (
    issue_flags["Total_Issue_Types"]
    .value_counts()
    .sort_index()
)

if not issue_counts.empty:
    issue_counts.plot(kind="bar")
    plt.title("Issue Overlap Across Rows")
    plt.xlabel("Number of Issue Types")
    plt.ylabel("Number of Rows")
else:
    plt.text(0.5, 0.5, "No Rows to Analyze",
             ha="center", va="center")
    plt.axis("off")

plt.tight_layout()
plt.savefig(
    os.path.join(BEFORE_DIR, "Issue_Overlap.png"),
    dpi=160
)
plt.show()
plt.close()

# ============================================================
# 11. REDUNDANT COLUMNS
# ============================================================

def find_redundant_columns(data):
    exact_duplicates = []
    highly_correlated = []

    # Exact duplicate columns
    for col1, col2 in combinations(data.columns, 2):
        if data[col1].equals(data[col2]):
            exact_duplicates.append({
                "Column_1": col1,
                "Column_2": col2
            })

    # Highly correlated numeric columns
    numeric = data.select_dtypes(include=[np.number])

    if numeric.shape[1] >= 2:
        correlation = numeric.corr().abs()
        columns = list(correlation.columns)

        for i, col1 in enumerate(columns):
            for col2 in columns[i + 1:]:
                value = correlation.loc[col1, col2]

                if pd.notna(value) and value >= 0.999:
                    highly_correlated.append({
                        "Column_1": col1,
                        "Column_2": col2,
                        "Correlation": value
                    })

    return (
        pd.DataFrame(
            exact_duplicates,
            columns=["Column_1", "Column_2"]
        ),
        pd.DataFrame(
            highly_correlated,
            columns=["Column_1", "Column_2", "Correlation"]
        )
    )

exact_duplicates, correlated_columns = find_redundant_columns(df)

exact_duplicates.to_csv(
    os.path.join(REPORT_DIR, "Exact_Duplicate_Columns.csv"),
    index=False
)

correlated_columns.to_csv(
    os.path.join(REPORT_DIR, "Highly_Correlated_Columns.csv"),
    index=False
)

print("Exact Duplicate Columns:")
display(exact_duplicates)

print("Highly Correlated Columns:")
display(correlated_columns)

# ============================================================
# OBJECTIVE 2: TIMELINE ANALYSIS
# ============================================================

# ============================================================
# 12. DATE ANALYSIS
# ============================================================

DATE_COLUMN = "issue_d"

def parse_dates(series):
    text_values = series.astype("string").str.strip()

    dates = pd.to_datetime(
        text_values,
        format="%b-%Y",
        errors="coerce"
    )

    remaining = dates.isna() & text_values.notna()

    if remaining.any():
        dates.loc[remaining] = pd.to_datetime(
            text_values.loc[remaining],
            errors="coerce"
        )

    return dates

if DATE_COLUMN in df.columns:
    dates = parse_dates(df[DATE_COLUMN])
    valid_dates = dates.notna()

    if valid_dates.any():
        timeline_data = df.loc[valid_dates].copy()

        timeline_data["Year"] = (
            dates.loc[valid_dates].dt.year.to_numpy()
        )

        timeline_data["Month"] = (
            dates.loc[valid_dates]
            .dt.to_period("M")
            .astype(str)
            .to_numpy()
        )

        timeline_data["_Row_Missing_Percentage"] = (
            timeline_data.isnull().mean(axis=1) * 100
        )

        # Monthly missingness
        monthly_nulls = (
            timeline_data.groupby("Month")[
                "_Row_Missing_Percentage"
            ]
            .mean()
            .reset_index(name="Missing_Percentage")
        )

        monthly_nulls.to_csv(
            os.path.join(REPORT_DIR, "Monthly_Nulls_Timeline.csv"),
            index=False
        )

        plt.figure(figsize=(15, 6))
        plt.plot(
            monthly_nulls["Month"],
            monthly_nulls["Missing_Percentage"],
            marker="o"
        )
        plt.xticks(rotation=90)
        plt.title("Monthly Missing Values Timeline")
        plt.xlabel("Month")
        plt.ylabel("Average Row Missing Percentage")
        plt.tight_layout()
        plt.savefig(
            os.path.join(BEFORE_DIR, "Monthly_Nulls_Timeline.png"),
            dpi=160
        )
        plt.show()
        plt.close()

        # Yearly row counts
        yearly_counts = (
            timeline_data.groupby("Year")
            .size()
            .reset_index(name="Row_Count")
        )

        yearly_counts.to_csv(
            os.path.join(REPORT_DIR, "Yearly_Row_Counts.csv"),
            index=False
        )

        display(yearly_counts)

        # Yearly missingness
        yearly_nulls = (
            timeline_data.groupby("Year")[
                "_Row_Missing_Percentage"
            ]
            .mean()
            .reset_index(name="Missing_Percentage")
        )

        plt.figure(figsize=(10, 5))
        plt.bar(
            yearly_nulls["Year"].astype(str),
            yearly_nulls["Missing_Percentage"]
        )
        plt.title("Yearly Missing Values")
        plt.xlabel("Year")
        plt.ylabel("Average Row Missing Percentage")
        plt.tight_layout()
        plt.savefig(
            os.path.join(BEFORE_DIR, "Yearly_Nulls.png"),
            dpi=160
        )
        plt.show()
        plt.close()

        # Top missing columns over time
        top_missing_columns = (
            df.isnull().mean()
            .sort_values(ascending=False)
            .head(8)
            .index.tolist()
        )

        monthly_column_nulls = (
            timeline_data.groupby("Month")[top_missing_columns]
            .apply(lambda x: x.isnull().mean() * 100)
        )

        plt.figure(figsize=(15, 7))

        for column in top_missing_columns:
            plt.plot(
                monthly_column_nulls.index,
                monthly_column_nulls[column],
                marker=".",
                label=column
            )

        plt.xticks(rotation=90)
        plt.title("Monthly Missingness of Top Columns")
        plt.ylabel("Missing Percentage")
        plt.legend(
            bbox_to_anchor=(1.02, 1),
            loc="upper left"
        )
        plt.tight_layout()
        plt.savefig(
            os.path.join(BEFORE_DIR, "Top_Columns_Timeline.png"),
            dpi=160
        )
        plt.show()
        plt.close()

        print("Valid dates:", int(valid_dates.sum()))
        print("Invalid or missing dates:", int((~valid_dates).sum()))

    else:
        print("No valid dates found in:", DATE_COLUMN)

else:
    print("Date column not found:", DATE_COLUMN)

# ============================================================
# CLEANING PHASE
# ============================================================

# ============================================================
# 13. DOCUMENTED CLEANING RULES
# ============================================================

# لا توجد قواعد حذف أو تعبئة أو تعديل معتمدة تلقائيًا.
# أضف القواعد فقط بعد مراجعة خطة المشروع.

APPROVED_RULES = [
    # مثال:
    # {
    #     "action": "strip_whitespace",
    #     "column": "some_column",
    #     "reason": "Documented and approved cleaning rule"
    # },

    # مثال:
    # {
    #     "action": "drop_exact_duplicate_rows",
    #     "reason": "Confirmed exact duplicate records"
    # }
]

# ============================================================
# 14. APPLY APPROVED RULES
# ============================================================

def apply_cleaning_rules(data, rules):
    cleaned = data.copy(deep=True)
    change_log = []

    for rule_number, rule in enumerate(rules, start=1):
        if "action" not in rule:
            raise ValueError(
                f"Rule {rule_number} is missing its action."
            )

        action = rule["action"]
        reason = str(rule.get("reason", "")).strip()

        if not reason:
            raise ValueError(
                "Every cleaning rule must have a documented reason."
            )

        if action == "strip_whitespace":
            column = rule.get("column")

            if column not in cleaned.columns:
                raise KeyError(column)

            is_text = (
                pd.api.types.is_object_dtype(cleaned[column])
                or pd.api.types.is_string_dtype(cleaned[column])
                or isinstance(cleaned[column].dtype, pd.CategoricalDtype)
            )

            if not is_text:
                raise TypeError(
                    f"strip_whitespace requires a text column: {column}"
                )

            before = cleaned[column].copy()
            mask = cleaned[column].notna()

            cleaned.loc[mask, column] = (
                cleaned.loc[mask, column]
                .astype("string")
                .str.strip()
            )

            changed = int(
                (
                    before.astype("string").fillna("<NULL>")
                    !=
                    cleaned[column].astype("string").fillna("<NULL>")
                ).sum()
            )

            change_log.append({
                "Rule_Number": rule_number,
                "Column": column,
                "Action": action,
                "Changed_Cells": changed,
                "Rows_Before": len(cleaned),
                "Rows_After": len(cleaned),
                "Reason": reason
            })

        elif action == "drop_exact_duplicate_rows":
            rows_before = len(cleaned)
            cleaned = cleaned.drop_duplicates().copy()

            change_log.append({
                "Rule_Number": rule_number,
                "Column": "ALL",
                "Action": action,
                "Changed_Cells": 0,
                "Rows_Before": rows_before,
                "Rows_After": len(cleaned),
                "Reason": reason
            })

        else:
            raise ValueError(
                f"Unsupported cleaning action: {action}"
            )

    log_columns = [
        "Rule_Number",
        "Column",
        "Action",
        "Changed_Cells",
        "Rows_Before",
        "Rows_After",
        "Reason"
    ]

    return cleaned, pd.DataFrame(
        change_log,
        columns=log_columns
    )

# Remove columns whose missingness is above the lower of:
# (1) the average missing percentage across all original columns, or
# (2) the fixed 39% ceiling.
column_missing_pct = df.isna().mean() * 100
average_missing_pct = float(column_missing_pct.mean())
missingness_cutoff = min(average_missing_pct, 39.0)
columns_to_drop = column_missing_pct[column_missing_pct > missingness_cutoff].index.tolist()

cleaning_base = df.drop(columns=columns_to_drop).copy(deep=True)

cleaned_df, cleaning_log = apply_cleaning_rules(
    cleaning_base,
    APPROVED_RULES
)

threshold_report = pd.DataFrame({
    "Column": column_missing_pct.index,
    "Missing_Percentage": column_missing_pct.values,
    "Average_Missing_Percentage": average_missing_pct,
    "Applied_Cutoff_Percentage": missingness_cutoff,
    "Removed": [c in columns_to_drop for c in column_missing_pct.index]
}).sort_values("Missing_Percentage", ascending=False)

threshold_report.to_csv(
    os.path.join(REPORT_DIR, "Missingness_Column_Removal_Report.csv"),
    index=False
)

if columns_to_drop:
    threshold_log = pd.DataFrame([{
        "Rule_Number": 0,
        "Column": ", ".join(columns_to_drop),
        "Action": "drop_high_missing_columns",
        "Changed_Cells": 0,
        "Rows_Before": len(df),
        "Rows_After": len(cleaned_df),
        "Reason": (
            f"Removed columns with missingness above {missingness_cutoff:.2f}%; "
            f"average missingness was {average_missing_pct:.2f}% and the cap is 39%."
        )
    }])
    cleaning_log = pd.concat([threshold_log, cleaning_log], ignore_index=True)

cleaned_df.to_csv(
    CLEANED_FILE,
    index=False
)

cleaning_log.to_csv(
    os.path.join(REPORT_DIR, "Cleaning_Log.csv"),
    index=False
)

display(cleaning_log)

# ============================================================
# 15. BEFORE AND AFTER COMPARISON
# ============================================================

comparison = pd.DataFrame({
    "Metric": [
        "Rows",
        "Columns",
        "Null Cells",
        "Duplicate Rows"
    ],
    "Before": [
        len(df),
        df.shape[1],
        int(df.isnull().sum().sum()),
        int(df.duplicated().sum())
    ],
    "After": [
        len(cleaned_df),
        cleaned_df.shape[1],
        int(cleaned_df.isnull().sum().sum()),
        int(cleaned_df.duplicated().sum())
    ]
})

display(comparison)

comparison.to_csv(
    os.path.join(REPORT_DIR, "Before_After_Comparison.csv"),
    index=False
)

# ============================================================
# 16. FINAL VALIDATION
# ============================================================

def validate_data(data):
    return {
        "Rows": int(len(data)),
        "Columns": int(data.shape[1]),
        "Null_Cells": int(data.isnull().sum().sum()),
        "Duplicate_Rows": int(data.duplicated().sum()),
        "Completely_Empty_Columns": int(data.isnull().all().sum()),
        "Constant_Columns": int(
            (data.nunique(dropna=False) <= 1).sum()
        )
    }

validation_results = pd.DataFrame([
    {
        "Dataset": "Before",
        **validate_data(df)
    },
    {
        "Dataset": "After",
        **validate_data(cleaned_df)
    }
])

display(validation_results)

validation_results.to_csv(
    os.path.join(REPORT_DIR, "Validation_Results.csv"),
    index=False
)

# ============================================================
# 17. EXPORT ALL RESULTS
# ============================================================

print("\n===================================")
print("DATA QUALITY PROCESS COMPLETED")
print("===================================")

print("Cleaned dataset:", CLEANED_FILE)
print("Reports:", REPORT_DIR)
print("Before-cleaning charts:", BEFORE_DIR)
print("After-cleaning outputs:", AFTER_DIR)

print("\nRaw data was not modified.")
print("Missing values were not filled with zero.")
print("No unapproved deletion or modification was performed.")

guide_text = f"""# Accepted Loans — Data Cleaning Guide

## Input and output
- Input: `data/Sampled/sample_accepted_2007_to_2018Q4.csv`
- Cleaned dataset: `data/Cleaned/accepted_cleaned.csv`
- Reports and charts: `docs/Data Cleaning/Accepted/`

## What this notebook does
1. Profiles the dataset, data types, duplicate rows, missing values, and memory usage.
2. Produces missing-value, co-missingness, outlier, category-consistency, issue-overlap, redundant-column, and date-timeline diagnostics.
3. Removes columns whose missing-value percentage is strictly greater than the lower of the dataset-wide average column missingness and 39%.
4. Applies only the additional rules explicitly listed in `APPROVED_RULES` (none are enabled by default).
5. Exports the cleaned CSV, cleaning log, before/after comparison, validation results, and missingness-based column-removal report.

## Important interpretation
- The raw sampled CSV is never overwritten.
- No missing values are imputed.
- The automatic removal rule applies to columns, not records.
- A column exactly at the cutoff is retained; only columns above the cutoff are removed.
- Review the removal report before analysis so that removed fields are understood.
"""
with open(GUIDE_FILE, "w", encoding="utf-8") as f:
    f.write(guide_text)

print("Cleaning guide:", GUIDE_FILE)
print("\\nALL STEPS FINISHED SUCCESSFULLY!")
